# Testing the MC-PILCO Functionality

In [1]:
# %load ~/dev/marthaler/header.py
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

%load_ext autoreload
%autoreload 2

In [2]:
# Enable Float64 for more stable matrix inversions.
import jax
import equinox as eqx
from jax import Array, config
import jax.numpy as jnp
import numpy as np
import jax.random as jr
from jaxtyping import ArrayLike, install_import_hook, Array, Float, Int, PyTree  
from typing import Optional
import matplotlib as mpl
import matplotlib.pyplot as plt

config.update("jax_enable_x64", True)

cols = mpl.rcParams["axes.prop_cycle"].by_key()["color"]

In [3]:
import gymnasium as gym

In [4]:
from jax_mc_pilco.controllers import Controller, RandomController, SumOfGaussians
from jax_mc_pilco.rewards import pendulum_cost#, cart_pole_cost
from jax_mc_pilco.model_learning.dynamical_models import IMGPR
from jax_mc_pilco.policy_learning.rollout import fit_controller
from jax_mc_pilco.simulators.simulation import remake_state, sample_from_environment

In [5]:
import optax as ox

In [6]:
from IPython import display

## Globals

In [7]:
num_particles = 400
num_trials = 8
T_sampling = 0.05
T_exploration = 0.35
T_control = 3.0
sim_timestep = 0.01
starting_dropout_probability = 0.25
control_horizon = int(T_control / T_sampling)
num_basis = 200
umax = 2.0

In [8]:
key = jr.key(42)

In [9]:
env = gym.make("Pendulum-v1")

In [10]:
action_dim = env.action_space.shape[0]
x, _ = env.reset()
state_dim = x.shape[0]
# state is cos_theta, sin_theta, theta_dot
timesteps = np.linspace(0, T_exploration, int(T_exploration / sim_timestep) + 1)

In [11]:
random_policy = RandomController(state_dim, action_dim, to_squash=True, max_action=umax)

control_policy = SumOfGaussians(
    state_dim,
    action_dim,
    num_basis,
    initial_log_lengthscales=None,
    initial_centers=None,
    to_squash=True,
    max_action=umax,
    key=key
)

# Test Rollout

In [12]:
states = []
actions = []
epsilon = 1e-4
exploration_policy = random_policy
num_opt_steps = 200
cosine_decay_scheduler = ox.cosine_decay_schedule(0.0001, decay_steps=num_opt_steps, alpha=0.95)


optimizer = ox.sgd(learning_rate=cosine_decay_scheduler)

key, subkey = jr.split(key) 
these_states, these_actions = sample_from_environment(env, timesteps, num_trials, exploration_policy, subkey)
states.extend(these_states)
actions.extend(these_actions)

In [13]:
import jax_mc_pilco

In [14]:
model = IMGPR(states=jnp.array(states),actions=jnp.array(actions),kernel_func=jax_mc_pilco.model_learning.gp.kernels.ExpSquared)
model.optimize()

In [15]:
model

IMGPR(
  mean_func=<function DynamicalModel.__init__.<locals>.<lambda>>,
  kernel_func=jax_mc_pilco.model_learning.gp.kernels.stationary.ExpSquared,
  training_data=f64[295,4],
  training_outputs=f32[295,3],
  num_outputs=3,
  input_dimension=4,
  num_datapoints=295,
  optimizers=[
    GradientTransformationExtraArgs(
      init=<function chain.<locals>.init_fn>,
      update=<function chain.<locals>.update_fn>
    ),
    GradientTransformationExtraArgs(
      init=<function chain.<locals>.init_fn>,
      update=<function chain.<locals>.update_fn>
    ),
    GradientTransformationExtraArgs(
      init=<function chain.<locals>.init_fn>,
      update=<function chain.<locals>.update_fn>
    )
  ],
  models=[
    {
      'kernel': {'log_coefficient': f64[], 'log_scale': f64[]},
      'likelihood': {'log_diag': f64[]},
      'mean': []
    },
    {
      'kernel': {'log_coefficient': f64[], 'log_scale': f64[]},
      'likelihood': {'log_diag': f64[]},
      'mean': []
    },
    {
      'ke

In [16]:
model.models[0]

{'kernel': {'log_coefficient': Array(-1.33811706, dtype=float64),
  'log_scale': Array(1.27345987, dtype=float64)},
 'likelihood': {'log_diag': Array(-1.58826871, dtype=float64)},
 'mean': []}

In [9]:
def test_policy(
    states: ArrayLike,
    timestep: Optional[Float] = None,
    key: Optional[ArrayLike] = None
)->Array:
    return jnp.ones_like(states)
    

In [25]:
class TestModel(eqx.Module):
    def get_samples(
        self,
        key: ArrayLike,
        samples: ArrayLike,
        actions: ArrayLike,
        num_samples: Int,
    )->Array:
        return samples
        

In [ ]:
def test_obj(
    
)